In [ ]:
import pandas as pd
from datetime import datetime
import dukascopy_python as dc
from pathlib import Path

instrument = "EUR/USD"   # change to "GBP/USD" for cable, "USD/JPY", etc.
year = 2025

# Save directly to OneDrive so the data is backed up and usable from other projects
base_dir = Path(r"C:\Users\benc1\OneDrive\claude\Tick Data")
symbol = instrument.replace("/", "")   # "EURUSD" -> used in filenames
out_dir = base_dir / "ticks"
out_dir.mkdir(parents=True, exist_ok=True)

# Download month by month, saving each to disk (resumable)
for month in range(1, 13):
    out = out_dir / f"{symbol}_{year}_{month:02d}.parquet"
    if out.exists():          # skip already-downloaded months
        print(f"Skipping {symbol} {year}-{month:02d} (already downloaded)")
        continue

    start = datetime(year, month, 1)
    # End is the first day of the next month (handles December correctly)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)

    print(f"Downloading {symbol} {start:%Y-%m}...")

    month_df = dc.fetch(
        instrument=instrument,
        interval=dc.INTERVAL_TICK,
        start=start,
        end=end,
        offer_side=dc.OFFER_SIDE_BID,
    )
    month_df.to_parquet(out)

# Combine all months from disk into one DataFrame
parts = sorted(out_dir.glob(f"{symbol}_{year}_*.parquet"))
df = pd.concat(pd.read_parquet(p) for p in parts)
print(f"\nFinished! Total Ticks: {len(df):,}")

# Save the combined result alongside the monthly files
df.to_parquet(base_dir / f"{symbol}_{year}_ticks.parquet")

## Pip / range bars vs raw ticks

A **range bar** closes every time price moves a fixed number of pips from its open — it is *event-driven*, not time-driven. Below we draw them as evenly-spaced **candlesticks** (a normal candle chart) and weave the raw tick line through them so you can see how each candle summarises the ticks underneath it.

Because the bars are event-based, the x-axis time labels are **unevenly spaced** — they bunch up in volatile periods (many candles form quickly) and spread out when the market is quiet. Smaller `N_PIPS` → more candles → more detail, but a whole day at 2 pips is too dense to read, so zoom to a few hours for that.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle, Patch
from matplotlib.collections import PatchCollection, LineCollection

# ---- config ----
PATH   = r"ticks\GBPUSD_2025_01.parquet"   # source tick file
PIP    = 0.0001    # GBP/USD: 1 pip = 0.0001  (use 0.01 for JPY pairs)
N_PIPS = 5         # <-- bar size in pips; try 2, 5, 10 (2 is very dense for a full day)
SIZE   = N_PIPS * PIP

ticks = pd.read_parquet(PATH)["bidPrice"]

# isolate the last calendar day present in the file
last_date = ticks.index[-1].normalize()
day = ticks[ticks.index.normalize() == last_date]
print(f"Last day: {last_date.date()}   ticks: {len(day):,}")


def build_range_bars(times, prices, size):
    """Close a new bar each time price moves `size` from the bar's open."""
    o = hi = lo = prices[0]
    t0 = times[0]
    bars = []
    for t, p in zip(times, prices):
        hi = max(hi, p); lo = min(lo, p)
        while p - o >= size:                 # up move -> close an up bar
            c = o + size
            bars.append((t0, t, o, max(hi, c), lo, c))
            o = hi = lo = c; t0 = t
        while o - p >= size:                 # down move -> close a down bar
            c = o - size
            bars.append((t0, t, o, hi, min(lo, c), c))
            o = hi = lo = c; t0 = t
    return pd.DataFrame(bars, columns=["t_start", "t_end", "open", "high", "low", "close"])


bars = build_range_bars(day.index.to_numpy(), day.to_numpy(float), SIZE)
print(f"{N_PIPS}-pip bars formed: {len(bars)}")
bars.head()

In [ ]:
# ---- plot: evenly-spaced range CANDLES with the raw tick line woven through ----
n = len(bars)
x = np.arange(n)
o = bars["open"].to_numpy();  c = bars["close"].to_numpy()
h = bars["high"].to_numpy();  l = bars["low"].to_numpy()
UP, DOWN = "#26a69a", "#ef5350"
colors = np.where(c >= o, UP, DOWN)
w = 0.7   # candle body width (in bar-index units)

fig, ax = plt.subplots(figsize=(15, 6))

# wicks (high-low) then bodies (open-close)
ax.add_collection(LineCollection([[(xi, li), (xi, hi)] for xi, li, hi in zip(x, l, h)],
                                 colors=list(colors), linewidths=0.9, zorder=1))
ax.add_collection(PatchCollection([Rectangle((xi - w / 2, min(oi, ci)), w, max(abs(ci - oi), 1e-9))
                                   for xi, oi, ci in zip(x, o, c)],
                                  facecolors=list(colors), edgecolors=list(colors), zorder=2))

# map each raw tick onto the candle (bar-index) axis so the line aligns with the candles
enum = mdates.date2num(np.concatenate([bars["t_start"].to_numpy()[:1], bars["t_end"].to_numpy()]))
tnum = mdates.date2num(day.index.to_numpy())
idx = np.clip(np.searchsorted(enum, tnum, side="right") - 1, 0, n - 1)
seg = enum[idx + 1] - enum[idx]; seg[seg == 0] = 1
tick_x = idx + np.clip((tnum - enum[idx]) / seg, 0, 1)
ax.plot(tick_x, day.to_numpy(), color="#1a1a1a", lw=0.5, alpha=0.75, zorder=3)

# x-axis: bar index, but labelled with the time each candle started (uneven = event bars)
ax.set_xlim(-1, n)
pos = np.linspace(0, n - 1, 9).astype(int)
ax.set_xticks(pos)
ax.set_xticklabels([pd.Timestamp(bars["t_start"].iloc[p]).strftime("%H:%M") for p in pos])
ax.set(title=f"GBP/USD  {last_date.date()}  —  {N_PIPS}-pip range candles ({n} bars) + raw ticks",
       ylabel="Price", xlabel="Time (UTC)")
ax.grid(True, alpha=0.2)
ax.legend(handles=[plt.Line2D([], [], color="#1a1a1a", lw=1, label="Raw bid ticks"),
                   Patch(color=UP, label=f"{N_PIPS}-pip up"),
                   Patch(color=DOWN, label=f"{N_PIPS}-pip down")], loc="upper left", fontsize=9)
plt.show()